# Day22 — chạy sạch Qwen3-4B trên Colab T4

Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**. Notebook chỉ chạy NB0–NB4 bằng `make pipeline` gốc, giữ 1.000 SFT và 800/100 preference. Hai judge chạy không NF4; ngưỡng sanity vẫn 80%. Không đảm bảo judge sẽ qua trước khi đo.

Không nhập API key. Kết quả mới được giữ riêng, không dùng báo cáo/số liệu 0.6B cũ. Sau khi chạy, tải ZIP và gửi về để cập nhật REFLECTION, kiểm tra gatekeeper và hoàn thiện bài. Nếu runtime bị ngắt, giữ log; khởi tạo session mới để chứng minh lượt chạy sạch.

## 1. Tạo nguồn sạch và môi trường mới

In [4]:
# Clean T4 setup: host Colab Python launches pinned Python 3.12 GPU jobs.
import os, sys, subprocess, json
from pathlib import Path
REPOSITORY = 'https://github.com/Neon310304/K4-L3-Track3-Day22-DPO-ORPO-Alignment.git'
SOURCE_COMMIT = '53a00376e779fbf06471dfeb42032027bfd56653'
WORK = Path('/content/day22-clean-t4')
if WORK.exists():
    assert not any((WORK / p).exists() for p in ['adapters/dpo', 'adapters/sft-mini', 'models/sft-merged', 'data/pref/train.parquet', 'submission/evidence/pipeline.json']), 'Training artifacts exist: use a fresh runtime.'
else:
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(WORK)], check=True)
subprocess.run(['git', 'fetch', 'origin', SOURCE_COMMIT], cwd=WORK, check=True)
sources = ['lab22', 'scripts', 'Makefile', 'requirements.txt', '.gitignore', '.gitattributes', 'notebooks/*.py', 'submission/screenshots/README.md']
subprocess.run(['git', 'checkout', SOURCE_COMMIT, '--', *sources], cwd=WORK, check=True)
subprocess.run(['git', 'update-ref', 'HEAD', SOURCE_COMMIT], cwd=WORK, check=True)
os.chdir(WORK)
os.environ.update({
    'COMPUTE_TIER': 'T4', 'BASE_MODEL': 'unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit',
    'MAX_LEN': '768', 'SFT_SLICE': '1000', 'PREF_TRAIN': '800', 'PREF_EVAL': '100',
    'DPO_BETA': '0.1', 'DPO_LR': '5e-6', 'DPO_EPOCHS': '1', 'SEED': '42',
    'GEN_MAX_NEW_TOKENS': '384', 'GEN_BATCH_SIZE': '1', 'JUDGE_PROMPTS': '50',
    'JUDGE_PROVIDER': 'rm', 'JUDGE_RM_4BIT': '0', 'JUDGE_RM_MAX_LENGTH': '4096',
    'JUDGE_RM_MODELS': 'Skywork/Skywork-Reward-V2-Qwen3-4B,Skywork/Skywork-Reward-V2-Llama-3.2-3B',
    'JUDGE_RM_DTYPE': 'float32', 'JUDGE_RM_DEVICE_MAP': 'auto', 'JUDGE_RM_MAX_GPU_GIB': '11', 'JUDGE_RM_MAX_CPU_GIB': '6',
    'DPO_SAVE_ON_CPU': '0', 'MPLBACKEND': 'Agg', 'TOKENIZERS_PARALLELISM': 'false',
    'HF_HOME': '/content/day22-hf-cache', 'HF_HUB_DISABLE_XET': '1',
})
Path('submission/evidence').mkdir(parents=True, exist_ok=True)
Path(os.environ['HF_HOME'], 'hub').mkdir(parents=True, exist_ok=True)
LAB_PY = '/content/day22-venv/bin/python'
with Path('submission/evidence/setup.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen([sys.executable, '-u', 'scripts/setup_t4_environment.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    assert process.wait() == 0, 'Environment setup failed; inspect setup.log.'
subprocess.run([LAB_PY, 'scripts/capture_environment.py'], check=True)
with Path('submission/evidence/pip-freeze.txt').open('w') as stream:
    subprocess.run([LAB_PY, '-m', 'pip', 'freeze'], stdout=stream, check=True)
smoke = subprocess.run([LAB_PY, 'scripts/verify.py', '--smoke'], capture_output=True, text=True)
Path('submission/evidence/smoke.log').write_text(smoke.stdout + smoke.stderr)
print(smoke.stdout, smoke.stderr, flush=True)
assert smoke.returncode == 0, 'Smoke failed; inspect smoke.log.'
print('SETUP PASSED. GPU jobs use:', LAB_PY, 'source:', SOURCE_COMMIT, flush=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 85.7 MB/s eta 0:00:00
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment with seed packages at: /content/day22-venv
 + pip==26.2.1
Activate with: source /content/day22-venv/bin/activate
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 32.1 MB/s  0:00:00
  Using cached torchao-0.13.0-1-cp39-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached unsloth-2026.10.2-py3-none-any.whl.metadata (82 kB)
  Using cached unsloth_zoo-2026.10.2-py3-none-any.whl.metadata (33 kB)
  Using cached trl-1.13.0-py3-none-any.whl.metadata (12 kB)
  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached lm_eval-0.4.13-py3-none-any.whl.metadata (64 kB)
  Using cached nvidia_cuda_nvrtc_cu

In [5]:
# Verify both real reward judges before starting training.
import os, subprocess, json
from pathlib import Path
assert Path(LAB_PY).exists(), 'Run setup first.'
assert not Path('adapters/dpo').exists(), 'Do not mix trained artifacts.'
preflight = {}
for label, name in [('qwen', 'Skywork/Skywork-Reward-V2-Qwen3-4B'), ('llama', 'Skywork/Skywork-Reward-V2-Llama-3.2-3B')]:
    output = Path('submission/evidence/judge-preflight-' + label + '.json')
    command = [LAB_PY, '-u', 'scripts/diagnose_reward_model.py', '--model', name, '--output', str(output)]
    with Path('submission/evidence/judge-preflight-' + label + '.log').open('w') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        exit_code = process.wait()
    assert exit_code == 0, 'Judge preflight failed: inspect saved log.'
    payload = json.loads(output.read_text())
    preflight[name] = payload['sanity_accuracy']
    assert payload['sanity_accuracy'] >= 0.8, 'Sanity stays at 80%; do not start training with this judge.'
print('PREFLIGHT PASSED:', preflight, flush=True)



Fetching 2 files: 100%|██████████| 2/2 [01:27<00:00, 43.83s/it]

Loading weights: 100%|██████████| 399/399 [00:29<00:00, 13.69it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
sanity 1/12: correct=True
sanity 2/12: correct=True
sanity 3/12: correct=True
sanity 4/12: correct=True
sanity 5/12: correct=True
sanity 6/12: correct=True
sanity 7/12: correct=True
sanity 8/12: correct=True
sanity 9/12: correct=True
sanity 10/12: correct=True
sanity 11/12: correct=True
sanity 12/12: correct=True
Sanity accuracy: 100.0%; passed=True

Fetching 2 files: 100%|██████████| 2/2 [01:37<00:00, 48.86s/it]

Loading weights: 100%|██████████| 255/255 [00:21<00:00, 11.73it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
sanity 1/12: correct=True
sanity 2/12: correct=True
sanity 3/12: correct=True
sanity 4/12: correct=True
sanity 5/12: correct=True
sanity 6/12: correct=True
sanity 7/12: correct=True
sanity 8/12: correct=True
sanity 9/12: cor

In [11]:
# Override Colab-only startup extensions in the isolated kernel.
specification = Path('/content/day22-venv/share/jupyter/kernels/python3/kernel.json')
kernel = json.loads(specification.read_text())
kernel['argv'].extend(['--IPKernelApp.extensions=[]', '--IPKernelApp.extra_extensions=[]'])
specification.write_text(json.dumps(kernel, indent=2) + chr(10))
probe = "from jupyter_client import KernelManager; k=KernelManager(kernel_name='python3'); k.start_kernel(); c=k.client(); c.start_channels(); c.wait_for_ready(timeout=30); print('ISOLATED KERNEL READY'); c.stop_channels(); k.shutdown_kernel(now=True)"
check = subprocess.run([LAB_PY, '-c', probe], env=job_environment, capture_output=True, text=True)
(evidence / 'kernel-smoke2.log').write_text(check.stdout + check.stderr)
print(check.stdout, check.stderr, flush=True)
assert check.returncode == 0, 'Kernel probe must pass before pipeline.'
print('Fresh workspace ready:', WORK, SOURCE_COMMIT, flush=True)


ISOLATED KERNEL READY
 [IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.

Fresh workspace ready: /content/day22-clean-t4 9fd6e21afddab86b1b2d92cef80329bcb47974dc


## 2. Chạy toàn bộ pipeline bắt buộc (có thể mất vài giờ)

In [ ]:
assert len(preflight) == 2 and all(value >= 0.8 for value in preflight.values()), 'Reward-model preflight must pass first.'
process = subprocess.Popen([LAB_PY, '-u', 'scripts/run_clean_pipeline.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='', flush=True)
PIPELINE_EXIT_CODE = process.wait()
print('make pipeline exit code:', PIPELINE_EXIT_CODE, flush=True)
print('Run the export cell to preserve actual artifacts and logs even if a stage failed.', flush=True)


Running make PY=/content/day22-venv/bin/python JUPYTEXT=/content/day22-venv/bin/python -m jupytext JUPYTER=/content/day22-venv/bin/python -m jupyter pipeline
[jupytext] Reading notebooks/00_dpo_loss_from_scratch.py in format py
[jupytext] Writing notebooks/00_dpo_loss_from_scratch.ipynb
[NbConvertApp] Converting notebook notebooks/00_dpo_loss_from_scratch.ipynb to notebook
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[NbConvertApp] Writing 12868 bytes to notebooks/00_dpo_loss_from_scratch.ipynb
[jupytext] Reading notebooks/01_sft_mini.py in format py
[jupytext] Writing notebooks/01_sft_mini.ipynb
[NbConvertApp] Converting notebook notebooks/01_sft_mini.ipynb to notebook
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communic

## 3. Kiểm trạng thái và tải ZIP bằng chứng (chạy cả khi pipeline lỗi)

In [ ]:
# Export real metadata and executed core notebooks; model weights stay in this runtime.
sys.path.insert(0, str(WORK / 'scripts'))
from build_clean_colab import render
export_source = ''.join(render()['cells'][-1]['source'])
exec(compile(export_source, 'day22-final-export', 'exec'))


{
  "pipeline_exit_code": 0,
  "fresh_artifacts": true,
  "base_model": "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
  "judge_nf4": false,
  "sanity": {
    "Skywork/Skywork-Reward-V2-Qwen3-4B": 0.0,
    "Skywork/Skywork-Reward-V2-Llama-3.2-3B": 1.0
  },
  "both_judges_pass": false,
  "reflection_status": "Needs update from this run; old 0.6B report was not copied."
}
If a judge still fails sanity, do not lower the threshold or claim the issue is fixed.
Saved /content/day22-t4-evidence.zip with 28 files; no .env/cache/model weights.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Also save this executed launcher notebook via File > Download > .ipynb.


In [2]:
# Diagnostic inspection: preserve the completed clean run.
from pathlib import Path
import os, sys, subprocess, json
from importlib import metadata
ROOT_DIAG = Path('/content/day22-clean-t4')
print('Runtime cwd:', Path.cwd(), 'workspace exists:', ROOT_DIAG.exists())
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
for package in ['torch', 'transformers', 'unsloth', 'trl']:
    try: print(package, metadata.version(package))
    except metadata.PackageNotFoundError: print(package, 'NOT INSTALLED')
for relative in ['submission/evidence/pipeline.json', 'data/eval/judge_summary.json', 'lab22/judge.py', 'notebooks/NB4_eval.py']:
    path = ROOT_DIAG / relative
    print('FILE:', relative, 'exists:', path.exists())
    if path.exists(): print(path.read_text()[:24000])
print('NOTEBOOK SOURCES:', [p.name for p in (ROOT_DIAG / 'notebooks').glob('*.py')])


Runtime cwd: /content workspace exists: False
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB

torch 2.11.0+cu130
transformers 5.18.0
unsloth NOT INSTALLED
trl NOT INSTALLED
FILE: submission/evidence/pipeline.json exists: False
FILE: data/eval/judge_summary.json exists: False
FILE: lab22/judge.py exists: False
FILE: notebooks/NB4_eval.py exists: False
NOTEBOOK SOURCES: []
